# RetainIQ — Phase 8.4: Benchmark Reporting & MySQL Publishing

## Objective

I publish the Phase 8 benchmark reference tables, benchmark gaps, and management views into MySQL so they can be reused by SQL, Power BI, and the later GenAI/RAG layers.

The publication layer preserves the analytical definitions from notebooks 8.1–8.3 rather than recomputing them inside MySQL.

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. Setup-only cells are not padded with artificial conclusions.

In [1]:
from getpass import getpass
from pathlib import Path
import pandas as pd
import numpy as np
import mysql.connector
from mysql.connector import Error
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

ROOT = Path("..")
OUTPUT_DIR = ROOT / "outputs"

MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "retainiq_user",
    "password": getpass("Enter MySQL password for retainiq_user: "),
    "database": "retainiq",
}

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

def print_result(message):
    print(f"Result: {message}")

In [2]:
required_outputs = {
    "benchmark_reference": "benchmark_reference.csv",
    "segment_benchmark": "segment_benchmark_gap.csv",
    "state_benchmark": "state_benchmark_gap.csv",
    "city_benchmark": "city_benchmark_gap.csv",
    "segment_summary": "segment_benchmark_status_summary.csv",
    "city_management": "city_benchmark_management_view.csv",
    "segment_strategy": "segment_strategy_benchmark_view.csv",
    "external_gap": "external_benchmark_gap.csv",
}

missing = [name for name, filename in required_outputs.items() if not (OUTPUT_DIR / filename).exists()]
if missing:
    raise FileNotFoundError(f"Missing required Phase 8 outputs: {missing}. Run notebooks 8.1–8.3 first.")

benchmark_reference = pd.read_csv(OUTPUT_DIR / required_outputs["benchmark_reference"])
segment_benchmark = pd.read_csv(OUTPUT_DIR / required_outputs["segment_benchmark"])
state_benchmark = pd.read_csv(OUTPUT_DIR / required_outputs["state_benchmark"])
city_benchmark = pd.read_csv(OUTPUT_DIR / required_outputs["city_benchmark"])
segment_summary = pd.read_csv(OUTPUT_DIR / required_outputs["segment_summary"])
city_management = pd.read_csv(OUTPUT_DIR / required_outputs["city_management"])
segment_strategy = pd.read_csv(OUTPUT_DIR / required_outputs["segment_strategy"])
external_gap = pd.read_csv(OUTPUT_DIR / required_outputs["external_gap"])

print_result("Loaded all Phase 8 benchmark outputs required for MySQL publication.")

Result: Loaded all Phase 8 benchmark outputs required for MySQL publication.


### Result & conclusion

The publication notebook has all benchmark artifacts available and ready to write to MySQL.

In [3]:
schema_sql = """
CREATE TABLE IF NOT EXISTS benchmark_reference (
    benchmark_scope VARCHAR(30) NOT NULL,
    benchmark_name VARCHAR(160) NOT NULL,
    metric VARCHAR(80) NOT NULL,
    benchmark_value DECIMAL(18,6) NULL,
    unit VARCHAR(80) NULL,
    source VARCHAR(255) NULL,
    source_date VARCHAR(50) NULL,
    population BIGINT NULL,
    notes TEXT NULL,
    PRIMARY KEY (benchmark_scope, benchmark_name, metric)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS segment_benchmark_gap (
    segment_id INT NOT NULL,
    segment_name VARCHAR(120) NOT NULL,
    retention_profile VARCHAR(80) NULL,
    customers INT NOT NULL,
    churned_customers INT NOT NULL,
    churn_rate_pct DECIMAL(8,2) NULL,
    avg_cltv DECIMAL(14,2) NULL,
    revenue_at_risk DECIMAL(16,2) NOT NULL,
    revenue_at_risk_per_customer DECIMAL(16,4) NULL,
    churn_gap_pp DECIMAL(10,4) NULL,
    cltv_gap_pct DECIMAL(12,4) NULL,
    revenue_risk_per_customer_gap_pct DECIMAL(12,4) NULL,
    benchmark_status VARCHAR(40) NULL,
    PRIMARY KEY (segment_id)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS state_benchmark_gap (
    state VARCHAR(100) NOT NULL,
    customers INT NOT NULL,
    churned_customers INT NOT NULL,
    churn_rate_pct DECIMAL(8,2) NULL,
    avg_cltv DECIMAL(14,2) NULL,
    revenue_at_risk DECIMAL(16,2) NOT NULL,
    revenue_at_risk_per_customer DECIMAL(16,4) NULL,
    churn_gap_pp DECIMAL(10,4) NULL,
    cltv_gap_pct DECIMAL(12,4) NULL,
    revenue_risk_per_customer_gap_pct DECIMAL(12,4) NULL,
    benchmark_status VARCHAR(40) NULL,
    PRIMARY KEY (state)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS city_benchmark_gap (
    state VARCHAR(100) NOT NULL,
    city VARCHAR(150) NOT NULL,
    customers INT NOT NULL,
    churned_customers INT NOT NULL,
    churn_rate_pct DECIMAL(8,2) NULL,
    avg_cltv DECIMAL(14,2) NULL,
    revenue_at_risk DECIMAL(16,2) NOT NULL,
    revenue_at_risk_per_customer DECIMAL(16,4) NULL,
    churn_gap_to_portfolio_pp DECIMAL(10,4) NULL,
    churn_gap_to_peer_city_pp DECIMAL(10,4) NULL,
    cltv_gap_pct DECIMAL(12,4) NULL,
    revenue_risk_per_customer_gap_pct DECIMAL(12,4) NULL,
    benchmark_status VARCHAR(50) NULL,
    PRIMARY KEY (state, city)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS benchmark_gap_summary (
    entity_type VARCHAR(30) NOT NULL,
    entities INT NOT NULL,
    above_benchmark INT NOT NULL,
    within_benchmark INT NOT NULL,
    below_benchmark INT NOT NULL,
    PRIMARY KEY (entity_type)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS city_benchmark_management_view (
    state VARCHAR(100) NOT NULL,
    city VARCHAR(150) NOT NULL,
    customers INT NOT NULL,
    churn_rate_pct DECIMAL(8,2) NULL,
    revenue_at_risk DECIMAL(16,2) NOT NULL,
    churn_gap_to_portfolio_pp DECIMAL(10,4) NULL,
    churn_gap_to_peer_city_pp DECIMAL(10,4) NULL,
    revenue_at_risk_per_customer DECIMAL(16,4) NULL,
    exposure_band VARCHAR(40) NULL,
    benchmark_exposure_flag VARCHAR(80) NULL,
    PRIMARY KEY (state, city)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS segment_strategy_benchmark_view (
    segment_id INT NOT NULL,
    segment_name VARCHAR(120) NOT NULL,
    retention_profile VARCHAR(80) NULL,
    customers INT NOT NULL,
    churn_rate_pct DECIMAL(8,2) NULL,
    avg_cltv DECIMAL(14,2) NULL,
    revenue_at_risk DECIMAL(16,2) NOT NULL,
    priority_score DECIMAL(18,2) NOT NULL,
    ease_of_intervention_score DECIMAL(6,2) NULL,
    priority_share_of_total_pct DECIMAL(10,4) NULL,
    churn_gap_pp DECIMAL(10,4) NULL,
    benchmark_status VARCHAR(40) NULL,
    revenue_risk_per_customer_gap_pct DECIMAL(12,4) NULL,
    cltv_gap_pct DECIMAL(12,4) NULL,
    PRIMARY KEY (segment_id)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS external_benchmark_gap (
    metric VARCHAR(80) NOT NULL,
    benchmark_name VARCHAR(160) NOT NULL,
    benchmark_value DECIMAL(18,6) NOT NULL,
    unit VARCHAR(80) NULL,
    source VARCHAR(255) NULL,
    source_date VARCHAR(50) NULL,
    population VARCHAR(255) NULL,
    notes TEXT NULL,
    internal_reference DECIMAL(18,6) NULL,
    gap DECIMAL(18,6) NULL,
    gap_definition VARCHAR(255) NULL
) ENGINE=InnoDB;
"""

connection = None
cursor = None
try:
    connection = get_connection()
    cursor = connection.cursor()
    for statement in [s.strip() for s in schema_sql.split(";") if s.strip()]:
        cursor.execute(statement)
    connection.commit()
finally:
    if cursor:
        cursor.close()
    if connection:
        connection.close()

print_result("Created the Phase 8 benchmark and management tables in MySQL.")

Result: Created the Phase 8 benchmark and management tables in MySQL.


### Result & conclusion

The MySQL reporting layer now has dedicated tables for benchmark references, grain-level gaps, and management views.

In [5]:
def clean_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value

def publish_table(table_name, delete_sql, insert_sql, rows):
    connection = None
    cursor = None
    try:
        connection = get_connection()
        cursor = connection.cursor()
        cursor.execute(delete_sql)
        cursor.executemany(insert_sql, rows)
        connection.commit()
    except Error as exc:
        if connection:
            connection.rollback()
        raise RuntimeError(f"Failed to publish {table_name}: {exc}") from exc
    finally:
        if cursor:
            cursor.close()
        if connection:
            connection.close()

if "source_date" not in benchmark_reference.columns:
    benchmark_reference["source_date"] = None

ref_cols = [
    "benchmark_scope", "benchmark_name", "metric", "benchmark_value",
    "unit", "source", "source_date", "population", "notes"
]
ref_rows = [tuple(clean_value(v) for v in row) for row in benchmark_reference[ref_cols].itertuples(index=False, name=None)]
publish_table(
    "benchmark_reference",
    "DELETE FROM benchmark_reference",
    "INSERT INTO benchmark_reference (benchmark_scope, benchmark_name, metric, benchmark_value, unit, source, source_date, population, notes) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
    ref_rows
)

seg_cols = ["segment_id","segment_name","retention_profile","customers","churned_customers","churn_rate_pct","avg_cltv","revenue_at_risk","revenue_at_risk_per_customer","churn_gap_pp","cltv_gap_pct","revenue_risk_per_customer_gap_pct","benchmark_status"]
seg_rows = [tuple(clean_value(v) for v in row) for row in segment_benchmark[seg_cols].itertuples(index=False, name=None)]
publish_table("segment_benchmark_gap", "DELETE FROM segment_benchmark_gap", "INSERT INTO segment_benchmark_gap VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", seg_rows)

state_cols = ["state","customers","churned_customers","churn_rate_pct","avg_cltv","revenue_at_risk","revenue_at_risk_per_customer","churn_gap_pp","cltv_gap_pct","revenue_risk_per_customer_gap_pct","benchmark_status"]
state_rows = [tuple(clean_value(v) for v in row) for row in state_benchmark[state_cols].itertuples(index=False, name=None)]
publish_table("state_benchmark_gap", "DELETE FROM state_benchmark_gap", "INSERT INTO state_benchmark_gap VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", state_rows)

city_cols = ["state","city","customers","churned_customers","churn_rate_pct","avg_cltv","revenue_at_risk","revenue_at_risk_per_customer","churn_gap_to_portfolio_pp","churn_gap_to_peer_city_pp","cltv_gap_pct","revenue_risk_per_customer_gap_pct","benchmark_status"]
city_rows = [tuple(clean_value(v) for v in row) for row in city_benchmark[city_cols].itertuples(index=False, name=None)]
publish_table("city_benchmark_gap", "DELETE FROM city_benchmark_gap", "INSERT INTO city_benchmark_gap VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", city_rows)

summary_df = pd.read_csv(OUTPUT_DIR / "benchmark_gap_summary.csv")
summary_cols = ["entity_type", "entities", "above_benchmark", "within_benchmark", "below_benchmark"]
summary_rows = [tuple(clean_value(v) for v in row) for row in summary_df[summary_cols].itertuples(index=False, name=None)]
publish_table("benchmark_gap_summary", "DELETE FROM benchmark_gap_summary", "INSERT INTO benchmark_gap_summary VALUES (%s,%s,%s,%s,%s)", summary_rows)

cmv_cols = ["state","city","customers","churn_rate_pct","revenue_at_risk","churn_gap_to_portfolio_pp","churn_gap_to_peer_city_pp","revenue_at_risk_per_customer","exposure_band","benchmark_exposure_flag"]
cmv_rows = [tuple(clean_value(v) for v in row) for row in city_management[cmv_cols].itertuples(index=False, name=None)]
publish_table("city_benchmark_management_view", "DELETE FROM city_benchmark_management_view", "INSERT INTO city_benchmark_management_view VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", cmv_rows)

ss_cols = ["segment_id","segment_name","retention_profile","customers","churn_rate_pct","avg_cltv","revenue_at_risk","priority_score","ease_of_intervention_score","priority_share_of_total_pct","churn_gap_pp","benchmark_status","revenue_risk_per_customer_gap_pct","cltv_gap_pct"]
ss_rows = [tuple(clean_value(v) for v in row) for row in segment_strategy[ss_cols].itertuples(index=False, name=None)]
publish_table("segment_strategy_benchmark_view", "DELETE FROM segment_strategy_benchmark_view", "INSERT INTO segment_strategy_benchmark_view VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", ss_rows)

ext_cols = ["metric","benchmark_name","benchmark_value","unit","source","source_date","population","notes","internal_reference","gap","gap_definition"]
ext_rows = [tuple(clean_value(v) for v in row) for row in external_gap[ext_cols].itertuples(index=False, name=None)] if not external_gap.empty else []
publish_table("external_benchmark_gap", "DELETE FROM external_benchmark_gap", "INSERT INTO external_benchmark_gap VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", ext_rows)

print_result("Published benchmark references, grain-level gaps, management views, and optional external gaps into MySQL.")



Result: Published benchmark references, grain-level gaps, management views, and optional external gaps into MySQL.


### Result & conclusion

The benchmark reporting layer is now persisted in MySQL. Re-running the notebook refreshes the tables from the latest Phase 8 CSV outputs.

In [6]:
view_sql = """
CREATE OR REPLACE VIEW vw_benchmark_reference AS SELECT * FROM benchmark_reference;
CREATE OR REPLACE VIEW vw_segment_benchmark_gap AS SELECT * FROM segment_benchmark_gap;
CREATE OR REPLACE VIEW vw_state_benchmark_gap AS SELECT * FROM state_benchmark_gap;
CREATE OR REPLACE VIEW vw_city_benchmark_gap AS SELECT * FROM city_benchmark_gap;
CREATE OR REPLACE VIEW vw_benchmark_gap_summary AS SELECT * FROM benchmark_gap_summary;
CREATE OR REPLACE VIEW vw_city_benchmark_management AS SELECT * FROM city_benchmark_management_view;
CREATE OR REPLACE VIEW vw_segment_strategy_benchmark AS SELECT * FROM segment_strategy_benchmark_view;
CREATE OR REPLACE VIEW vw_external_benchmark_gap AS SELECT * FROM external_benchmark_gap;
"""

connection = None
cursor = None
try:
    connection = get_connection()
    cursor = connection.cursor()
    for statement in [s.strip() for s in view_sql.split(";") if s.strip()]:
        cursor.execute(statement)
    connection.commit()
finally:
    if cursor:
        cursor.close()
    if connection:
        connection.close()

print_result("Created the Phase 8 MySQL reporting views for SQL and BI consumption.")

Result: Created the Phase 8 MySQL reporting views for SQL and BI consumption.


### Result & conclusion

The tables now have stable view-based endpoints for downstream BI and later GenAI/RAG retrieval.

In [7]:
validation_query = """
SELECT
    (SELECT COUNT(*) FROM benchmark_reference) AS benchmark_reference_rows,
    (SELECT COUNT(*) FROM segment_benchmark_gap) AS segment_gap_rows,
    (SELECT COUNT(*) FROM state_benchmark_gap) AS state_gap_rows,
    (SELECT COUNT(*) FROM city_benchmark_gap) AS city_gap_rows,
    (SELECT COUNT(*) FROM benchmark_gap_summary) AS benchmark_summary_rows,
    (SELECT COUNT(*) FROM city_benchmark_management_view) AS city_management_rows,
    (SELECT COUNT(*) FROM segment_strategy_benchmark_view) AS segment_strategy_rows,
    (SELECT COUNT(*) FROM external_benchmark_gap) AS external_gap_rows;
"""

connection = None
cursor = None
try:
    connection = get_connection()
    cursor = connection.cursor(dictionary=True)
    cursor.execute(validation_query)
    validation = pd.DataFrame(cursor.fetchall())
finally:
    if cursor:
        cursor.close()
    if connection:
        connection.close()

display(validation)
print_result("Validated that the Phase 8 MySQL reporting objects contain published rows; exact counts follow the executed Phase 8 outputs.")

,benchmark_reference_rows,segment_gap_rows,state_gap_rows,city_gap_rows,benchmark_summary_rows,city_management_rows,segment_strategy_rows,external_gap_rows
0,6,3,1,27,3,27,3,0


Result: Validated that the Phase 8 MySQL reporting objects contain published rows; exact counts follow the executed Phase 8 outputs.


### Result & conclusion

The final validation confirms that the benchmark publication objects are populated after the write and view-creation steps.

In [8]:
benchmark_reference.to_csv(OUTPUT_DIR / "benchmark_reference.csv", index=False)
segment_benchmark.to_csv(OUTPUT_DIR / "segment_benchmark_gap.csv", index=False)
state_benchmark.to_csv(OUTPUT_DIR / "state_benchmark_gap.csv", index=False)
city_benchmark.to_csv(OUTPUT_DIR / "city_benchmark_gap.csv", index=False)
segment_summary.to_csv(OUTPUT_DIR / "segment_benchmark_status_summary.csv", index=False)
city_management.to_csv(OUTPUT_DIR / "city_benchmark_management_view.csv", index=False)
segment_strategy.to_csv(OUTPUT_DIR / "segment_strategy_benchmark_view.csv", index=False)
external_gap.to_csv(OUTPUT_DIR / "external_benchmark_gap.csv", index=False)

print_result("Refreshed the portable Phase 8 CSV outputs after successful MySQL publication.")

Result: Refreshed the portable Phase 8 CSV outputs after successful MySQL publication.


### Result & conclusion

Phase 8 is now available in both portable CSV form and persistent MySQL reporting form. These outputs provide the benchmark context that can be handed into the GenAI/RAG phases.